In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F, Window

@dp.table(name="dev.silver.clientes_validos", comment="Clientes tipados e válidos para CDC")
@dp.expect_or_drop("id_cliente_obrigatorio", "id_cliente IS NOT NULL")
@dp.expect_or_drop("data_atualizacao_obrigatoria", "data_atualizacao IS NOT NULL")
@dp.expect("renda_nao_negativa", "renda IS NULL OR CAST(renda AS DOUBLE) >= 0")
def slv_clientes_validos():
    return (spark.readStream.table("dev.bronze.clientes")
      .select(
        F.col("id_cliente").cast("long").alias("id_cliente"),
        F.col("cpf").cast("string").alias("cpf"),
        F.col("nome").cast("string").alias("nome"),
        F.col("cidade").cast("string").alias("cidade"),
        F.col("estado").cast("string").alias("estado"),
        F.col("renda").cast("decimal(18,2)").alias("renda"),
        F.col("segmento").cast("string").alias("segmento"),
        F.to_timestamp("data_atualizacao").alias("data_atualizacao"),
        F.upper(F.col("operacao")).alias("operacao"),
        "timestamp_ingestao"
      ))

@dp.table(name="dev.silver.clientes_quarentena", comment="Clientes com campos/chaves inválidas")
def slv_clientes_quarentena():
    df = spark.readStream.table("dev.bronze.clientes")
    return df.filter(
        F.col("id_cliente").isNull() |
        F.col("data_atualizacao").isNull() |
        (F.col("renda").cast("double") < 0) |
        F.col("cpf").isNull() |
        (F.trim(F.col("nome")) == "")
    ).withColumn("motivo_quarentena", F.lit("cliente inválido"))

@dp.table(name="dev.silver.contas_validas", comment="Contas tipadas")
@dp.expect_or_drop("id_conta_obrigatorio", "id_conta IS NOT NULL")
@dp.expect_or_drop("id_cliente_obrigatorio", "id_cliente IS NOT NULL")
@dp.expect_or_drop("data_atualizacao_obrigatoria", "data_atualizacao IS NOT NULL")
def slv_contas_validas():
    return (spark.readStream.table("dev.bronze.contas")
      .select(
        F.col("id_conta").cast("long").alias("id_conta"),
        F.col("id_cliente").cast("long").alias("id_cliente"),
        "tipo_conta", "status_conta",
        F.to_date("data_abertura").alias("data_abertura"),
        F.to_timestamp("data_atualizacao").alias("data_atualizacao"),
        F.upper(F.col("operacao")).alias("operacao"),
        "timestamp_ingestao"
      ))

@dp.table(name="dev.silver.contas_quarentena", comment="Contas inválidas")
def slv_contas_quarentena():
    df = spark.readStream.table("dev.bronze.contas")
    return df.filter(F.col("id_conta").isNull() | F.col("id_cliente").isNull() |
                     F.col("data_atualizacao").isNull()).withColumn(
                         "motivo_quarentena", F.lit("conta inválida"))

@dp.table(name="dev.silver.cartoes_validos", comment="Cartões tipados")
@dp.expect_or_drop("id_cartao_obrigatorio", "id_cartao IS NOT NULL")
@dp.expect_or_drop("id_conta_obrigatorio", "id_conta IS NOT NULL")
@dp.expect("limite_nao_negativo", "limite IS NULL OR CAST(limite AS DOUBLE) >= 0")
def slv_cartoes_validos():
    return (spark.readStream.table("dev.bronze.cartoes")
      .select(
        F.col("id_cartao").cast("long").alias("id_cartao"),
        F.col("id_conta").cast("long").alias("id_conta"),
        "tipo_cartao", F.col("limite").cast("decimal(18,2)").alias("limite"),
        "status_cartao", F.to_timestamp("data_atualizacao").alias("data_atualizacao"),
        F.upper(F.col("operacao")).alias("operacao"), "timestamp_ingestao"
      ))

@dp.table(name="dev.silver.cartoes_quarentena", comment="Cartões inválidos")
def slv_cartoes_quarentena():
    df = spark.readStream.table("dev.bronze.cartoes")
    return df.filter(F.col("id_cartao").isNull() | F.col("id_conta").isNull() |
                     (F.col("limite").cast("double") < 0)).withColumn(
                         "motivo_quarentena", F.lit("cartão inválido"))

@dp.table(name="dev.silver.transacoes_validas", comment="Transações deduplicadas e tipadas")
@dp.expect_or_drop("id_transacao_obrigatorio", "id_transacao IS NOT NULL")
@dp.expect_or_drop("id_cartao_obrigatorio", "id_cartao IS NOT NULL")
@dp.expect_or_drop("data_transacao_obrigatoria", "data_transacao IS NOT NULL")
@dp.expect_or_drop("valor_positivo", "CAST(valor AS DECIMAL(18,2)) > 0")
def slv_transacoes_validas():
    df = spark.readStream.table("dev.bronze.transacoes")
    return df.select(
        F.col("id_transacao").cast("long").alias("id_transacao"),
        F.col("id_cartao").cast("long").alias("id_cartao"),
        F.to_timestamp("data_transacao").alias("data_transacao"),
        F.col("valor").cast("decimal(18,2)").alias("valor"),
        "mcc", "estabelecimento", "canal", "pais", "moeda",
        "timestamp_ingestao"
    )

@dp.table(name="dev.silver.transacoes_quarentena", comment="Transações inválidas")
def slv_transacoes_quarentena():
    df = spark.readStream.table("dev.bronze.transacoes")
    return df.filter(F.col("id_transacao").isNull() | F.col("id_cartao").isNull() |
                     F.col("data_transacao").isNull() |
                     (F.col("valor").cast("decimal(18,2)") <= 0)).withColumn(
                         "motivo_quarentena", F.lit("transação inválida"))

@dp.table(name="dev.silver.eventos_risco", comment="Eventos de risco tipados")
@dp.expect_or_drop("id_evento_obrigatorio", "id_evento IS NOT NULL")
@dp.expect_or_drop("id_transacao_obrigatorio", "id_transacao IS NOT NULL")
def slv_eventos_risco():
    return (spark.readStream.table("dev.bronze.eventos_risco")
      .select(F.col("id_evento").cast("long").alias("id_evento"),
              F.col("id_transacao").cast("long").alias("id_transacao"),
              "tipo_evento", "severidade",
              F.to_timestamp("data_evento").alias("data_evento")))

@dp.table(name="dev.silver.estornos", comment="Estornos tipados")
@dp.expect_or_drop("id_estorno_obrigatorio", "id_estorno IS NOT NULL")
@dp.expect_or_drop("id_transacao_obrigatorio", "id_transacao IS NOT NULL")
def slv_estornos():
    df = spark.readStream.table("dev.bronze.estornos")
    return df.select(F.col("id_estorno").cast("long").alias("id_estorno"),
                     F.col("id_transacao").cast("long").alias("id_transacao"),
                     F.to_timestamp("data_estorno").alias("data_estorno"),
                     "motivo",
                     F.col("valor_estorno").cast("decimal(18,2)").alias("valor_estorno"))
